# Tier 1a — B0 vs B-gold on Kaggle (LifeMate only)

**Purpose.** Measure, on one frozen stratified dev set, how much of the
task error is ASR error and how much is LLM/agent error, by running
the same samples and the same LLM twice: once with the baseline ASR
transcript as the current-turn input (B0), once with the gold
transcript as the current-turn input (B-gold, a real `AgentLoop.run()`,
never replay-gold). History is fixed in both modes; only the current
turn varies. G&G is shadow-only throughout: observer ON to log, never
to block. `pending_ask_stop` stays False, history V2 stays off.

**Source pin (fail-fast, checked in the preflight cell before any
weight is loaded).** Source tree: this branch, `muse/chat3/p1-tier1a-fix` (Phase 1 work; the coordinator merges and pushes the integration branch later, so do not clone `tier1a-kaggle` for a Phase 1 run — it lacks these fixes).
`lifemate/` must equal the CPU pin `ddbe0a0b18a6fb9f6e7d261628a51e7a2c6c10c6`
(`git diff ddbe0a0 HEAD -- lifemate/` empty). Config:
`lifemate/configs/recommended.json`, sha256
`71d5f38f…e980076` (full hash in the preflight cell). Split:
`lifemate/splits/train_dev_heldout.json`, sha256
`eefd0faa…18280aa`, dev 1237 / held-out 308. LLM
`Qwen/Qwen3-4B-Instruct-2507` rev `cdbee75`, temperature 0.0, seed 42,
context 16384, `max_steps` 12, `max_repeats` 2, grounding on.
ASR `vinai/PhoWhisper-small`, faster-whisper, greedy, language `vi`.
Total 4.244B / LLM 4.0B, under the organiser limits 6B / 5B.

**WARNING — provisional set excluded.** An earlier hash-only 200-id
set exists in the executor's working directory, but it was never
stratified (no `train.jsonl` was available) and it is NOT in this
branch and NOT accepted here. This notebook STOPS before loading
weights unless a stratified-frozen manifest exists or is created by
the refreeze cell from real data. No provisional ids are used, quoted,
or reported anywhere in this notebook.

**Run / score separation.** The run phase never reads gold tool calls
or gold state hashes (proved by the `check-separation` step in the
preflight cell). Gold calls are read only by the score phase, after
run traces are already on disk. B-gold uses only the gold *transcript*
as agent input; gold calls are never input or history.

**Sensitive outputs stay private.** Every output of this notebook
(traces, transcripts, caches, reports) goes to `/kaggle/working`
only. Nothing is written back into the repo, and no data, transcript,
weight, or trace is committed.

**What this notebook does NOT do.** No GPU training, no submission,
no contact with the organisers, no held-out use, no model result
claim until the score cell actually runs. If any preflight, manifest,
or OFF-vs-ON check fails, the notebook stops at that cell.

**Phase 1 fix (this branch).** faster-whisper CANNOT load
`vinai/PhoWhisper-small` directly: the checkpoint is
transformers-format (no `model.bin` for CTranslate2). The preflight
cell now checks the ASR model files before any weight is loaded
and stops with the conversion command if the direct load is
impossible. Either convert once and set `TIER1A_ASR_MODEL_DIR` to
the converted directory, or set `TIER1A_ASR_BACKEND=transformers`
for the same checkpoint via transformers (recorded in every run
manifest as `asr_backend_actual`). The dependency cell installs
missing backends (check-before, install, re-check) and records
installed versions. Run cells use `--resume`, and the score cell
refuses to print any total unless both arms cover exactly the
manifest ids and the OFF-vs-ON pilot (20) is identical.

This notebook now has 10 cells (0 purpose, 1 preflight,
2 dependencies, 3 refreeze, 4 ASR cache, 5 run B0, 6 run B-gold,
7 OFF-vs-ON, 8 score, 9 list artifacts).


In [ ]:
# Cell 1 — Preflight (FAIL-FAST, BEFORE any weight is loaded).
# Stdlib + git only. No torch, no vllm, no faster-whisper is imported here.
import hashlib, json, os, subprocess, sys
from pathlib import Path

def sha256_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

def run_cmd(cmd, what):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{what} FAILED (rc={r.returncode}):\n{r.stdout}\n{r.stderr}")
    return r

# --- Parameters: override with Kaggle environment variables. ---
REPO_URL = os.environ.get("TIER1A_REPO_URL", "https://github.com/lenamkhanhh/ZaloAIC.git")
BRANCH = os.environ.get("TIER1A_BRANCH", "muse/chat3/p1-tier1a-fix")  # Phase 1 branch; override only for the integration branch after the coordinator merges it
REPO_DIR = Path(os.environ.get("TIER1A_SOURCE_TREE", "/kaggle/working/ZaloAIC"))
# DATA_DIR must be the EXTRACTED train+public_test root, i.e. the directory
# whose train/ subdirectory contains train.jsonl and the wav files.
# On Kaggle, point TIER1A_DATA at your mounted dataset root; the default
# /kaggle/input is correct only if that root is mounted directly there.
DATA_DIR = Path(os.environ.get("TIER1A_DATA", "/kaggle/input"))
WORKING = Path(os.environ.get("TIER1A_WORKING", "/kaggle/working"))
OUT = WORKING / "tier1a-output"
OUT.mkdir(parents=True, exist_ok=True)
RUNNER_DIR = REPO_DIR / "analysis" / "tier1a"
RUNNER = RUNNER_DIR / "tier1a_runner.py"
REFREEZE = RUNNER_DIR / "refreeze.py"
MANIFEST = Path(os.environ.get("TIER1A_BENCHMARK", str(WORKING / "benchmark-stratified-frozen.json")))
ASR_BACKEND = os.environ.get("TIER1A_ASR_BACKEND", "faster-whisper")
ASR_MODEL_DIR_RAW = os.environ.get("TIER1A_ASR_MODEL_DIR")
ASR_MODEL_DIR = Path(ASR_MODEL_DIR_RAW) if ASR_MODEL_DIR_RAW else None
print(f"ASR_BACKEND={ASR_BACKEND} ASR_MODEL_DIR={ASR_MODEL_DIR}")
print(f"REPO_DIR={REPO_DIR}\nDATA_DIR={DATA_DIR}\nOUT={OUT}\nMANIFEST={MANIFEST}")

# --- Clone the branch if the source tree is absent. ---
if not (REPO_DIR / "lifemate" / "configs" / "recommended.json").exists():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError(f"{REPO_DIR} exists but has no lifemate/ tree: remove it and rerun so the clone is clean.")
    run_cmd(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], "git clone branch")
    print("Cloned branch into", REPO_DIR)
assert RUNNER.exists(), f"runner missing: {RUNNER}"
assert REFREEZE.exists(), f"refreeze missing: {REFREEZE}"

# --- Source / config / split SHAs and config fields. ---
EXPECTED_CONFIG_SHA = "71d5f38f23d126427b08c09925cbbd9da933fc3750cbec55942809737e980076"
EXPECTED_SPLIT_SHA = "eefd0faa163285b23d2634bc4cc04f515b58367a2344599e46e85d48818280aa"
cfg_path = REPO_DIR / "lifemate" / "configs" / "recommended.json"
split_path = REPO_DIR / "lifemate" / "splits" / "train_dev_heldout.json"
cfg_sha, split_sha = sha256_file(cfg_path), sha256_file(split_path)
assert cfg_sha == EXPECTED_CONFIG_SHA, f"config SHA mismatch: {cfg_sha}"
assert split_sha == EXPECTED_SPLIT_SHA, f"split SHA mismatch: {split_sha}"
cfg = json.loads(cfg_path.read_text())
checks = {
    "llm_checkpoint": cfg["llm_checkpoint"] == "Qwen/Qwen3-4B-Instruct-2507",
    "llm_revision": cfg["llm_revision"] == "cdbee75",
    "llm_temperature": cfg["llm_temperature"] == 0.0,
    "llm_seed": cfg["llm_seed"] == 42,
    "llm_max_context_tokens": cfg["llm_max_context_tokens"] == 16384,
    "asr_checkpoint": cfg["asr_checkpoint"] == "vinai/PhoWhisper-small",
    "asr_backend": cfg["asr_backend"] == "faster-whisper",
    "asr_greedy": cfg["asr_greedy"] is True,
    "asr_language": cfg["asr_language"] == "vi",
    "grounding": cfg["grounding"] is True,
    "max_steps": cfg["max_steps"] == 12,
    "max_repeats": cfg["max_repeats"] == 2,
    "total_params_b_lt_6": cfg["total_params_b"] < 6.0,
    "llm_params_b_lt_5": cfg["llm_params_b"] < 5.0,
}
assert all(checks.values()), f"config field check failed: {checks}"
print("Config checks:", checks)
print("ASR revision note: recommended.json records no ASR revision field; the ASR cache "
      "meta therefore uses revision='' and the checkpoint string above is the pin. "
      "If you set an ASR revision elsewhere, record it in the run report — do not silently "
      "substitute a different checkpoint.")

# --- Git source check: lifemate/ must equal the ddbe0a0 tree. ---
diff = run_cmd(["git", "-C", str(REPO_DIR), "diff", "ddbe0a0", "HEAD", "--", "lifemate/"], "git lifemate diff")
head = run_cmd(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], "git rev-parse HEAD")
anc = subprocess.run(["git", "-C", str(REPO_DIR), "merge-base", "--is-ancestor", "ddbe0a0", "HEAD"])
assert diff.stdout.strip() == "", "lifemate/ differs from ddbe0a0 — STOP, do not run."
assert anc.returncode == 0, "ddbe0a0 is not an ancestor of HEAD — STOP."
print("Source git check: HEAD =", head.stdout.strip(), "| lifemate diff empty | ddbe0a0 ancestor: True")

# --- Split membership counts (id-only file, no labels read). ---
split = json.loads(split_path.read_text())
dev_n = sum(1 for v in split.values() if v == "dev")
held_n = sum(1 for v in split.values() if v == "heldout")
assert (dev_n, held_n) == (1237, 308), f"split counts changed: dev={dev_n} heldout={held_n}"
print(f"Split: dev={dev_n} heldout={held_n}")

# --- Run/score separation in code (stdlib, before weights). ---
run_cmd([sys.executable, str(RUNNER), "check-separation"], "check-separation")
print("Run/score separation: PASS (no gold-call read in the run path)")

# --- Input size, before weights: data root, wav files, total bytes. ---
train_jsonl = DATA_DIR / "train" / "train.jsonl"
if not train_jsonl.exists():
    candidates = []
    if DATA_DIR.exists():
        candidates = [str(p) for p in sorted(DATA_DIR.rglob("train.jsonl"))[:10]]
    raise RuntimeError(
        f"no train.jsonl under {DATA_DIR}. Set TIER1A_DATA to the EXTRACTED "
        f"train+public_test root, i.e. the directory whose train/ subdirectory "
        f"contains train.jsonl (not the zip, not a parent of the extracted folder). "
        f"train.jsonl candidates found under DATA_DIR: {candidates} — STOP.")
wavs = sorted((DATA_DIR / "train").rglob("*.wav"))
wav_bytes = sum(p.stat().st_size for p in wavs)
print(f"Input: train.jsonl={train_jsonl} ({train_jsonl.stat().st_size} bytes) | "
      f"wav files={len(wavs)} | wav bytes={wav_bytes}")
assert len(wavs) > 0, "no wav files under DATA_DIR/train — STOP."

# --- ASR model file check, still before weights (Phase 1). ---
# Fails fast with the conversion command when a direct faster-whisper
# load of vinai/PhoWhisper-small is impossible; no weight is downloaded here.
asr_cmd = [sys.executable, str(RUNNER), "check-asr-model", "--asr-backend", ASR_BACKEND]
if ASR_MODEL_DIR is not None:
    asr_cmd += ["--asr-model-dir", str(ASR_MODEL_DIR)]
asr_r = subprocess.run(asr_cmd, capture_output=True, text=True)
print(asr_r.stdout[-2000:])
if asr_r.returncode != 0:
    raise RuntimeError(
        f"ASR model check BLOCKED (backend={ASR_BACKEND}, model_dir={ASR_MODEL_DIR}):\n"
        f"{asr_r.stdout}\n{asr_r.stderr}\n"
        "Fix with the conversion command or --asr-backend transformers "
        "(see README Phase 1). STOP before weights.")
try:
    asr_check = json.loads(asr_r.stdout)
except Exception:
    asr_check = {"raw": asr_r.stdout}
print("ASR model check PASS:", asr_check)

preflight = {
    "head": head.stdout.strip(), "config_sha256": cfg_sha, "split_sha256": split_sha,
    "asr_backend": ASR_BACKEND,
    "asr_model_dir": str(ASR_MODEL_DIR) if ASR_MODEL_DIR else None,
    "asr_check": asr_check,
    "config_checks": checks, "lifemate_diff_empty": True, "ddbe0a0_ancestor": True,
    "split_dev": dev_n, "split_heldout": held_n, "run_gold_separation": "PASS",
    "data_dir": str(DATA_DIR), "wav_files": len(wavs), "wav_bytes": wav_bytes,
    "weights_loaded": False, "note": "Preflight ran before any weight load.",
}
(OUT / "preflight_report.json").write_text(json.dumps(preflight, indent=2), encoding="utf-8")
print("Preflight PASS ->", OUT / "preflight_report.json")
print("Next: refreeze cell (creates or validates the frozen manifest, still before weights).")


In [ ]:
# Cell 2 — Dependencies: check-before, install-if-missing, re-check.
# No weights are loaded here, and no heavy module is imported: versions
# are read from package metadata only. If a backend is missing, it is
# installed with pip (faster-whisper is pinned to 1.2.1, the version
# recorded in RUN_LIFEMATE.md section 5; vllm / transformers /
# accelerate / ctranslate2 have no version pin recorded in the repo,
# so they are installed unpinned and their installed versions are
# recorded in dependency_report.json — never invented). If the install
# still leaves a backend missing, this cell raises BLOCKED with the
# exact manual pip command instead of starting a partial run.
import importlib.metadata, json, subprocess, sys
from pathlib import Path

DIST_FOR_MODULE = {
    "torch": "torch", "vllm": "vllm", "faster_whisper": "faster-whisper",
    "transformers": "transformers", "accelerate": "accelerate",
    "ctranslate2": "ctranslate2",
}
PIP_FOR_DIST = {"faster-whisper": "faster-whisper==1.2.1"}

def versions():
    out = {}
    for mod, dist in DIST_FOR_MODULE.items():
        try:
            out[mod] = importlib.metadata.version(dist)
        except Exception:
            out[mod] = None
    return out

before = versions()
missing = [m for m, v in before.items() if v is None]
print("Dependencies before:", before)
if missing:
    pkgs = [PIP_FOR_DIST.get(DIST_FOR_MODULE[m], DIST_FOR_MODULE[m]) for m in missing]
    print(f"Missing backends {missing} — installing: pip install {' '.join(pkgs)}")
    r = subprocess.run([sys.executable, "-m", "pip", "install"] + pkgs,
                       capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        raise RuntimeError(
            f"BLOCKED: pip install failed (rc={r.returncode}):\n{r.stderr[-3000:]}\n"
            f"Install manually, then rerun this cell:\n"
            f"  pip install {' '.join(pkgs)}")
after = versions()
still_missing = [m for m, v in after.items() if v is None]
if still_missing:
    pkgs = [PIP_FOR_DIST.get(DIST_FOR_MODULE[m], DIST_FOR_MODULE[m]) for m in still_missing]
    raise RuntimeError(
        f"BLOCKED: backends still missing after install: {still_missing}. "
        f"Install manually, then rerun this cell:\n  pip install {' '.join(pkgs)}")
dep_report = {"before": before, "after": after, "weights_loaded": False,
              "asr_backend": ASR_BACKEND,
              "asr_model_dir": str(ASR_MODEL_DIR) if ASR_MODEL_DIR else None}
(OUT / "dependency_report.json").write_text(json.dumps(dep_report, indent=2), encoding="utf-8")
print("Dependencies PASS ->", OUT / "dependency_report.json")
print("Installed versions:", after)
print("Next: refreeze cell (still before weights).")

In [ ]:
# Cell 2 — Refreeze (only if no frozen manifest yet; else skip with log).
# Still before any weight is loaded. Refreeze reads train.jsonl labels
# ONLY to stratify and check coverage; its output is ids + counts.
import json, subprocess, sys
from pathlib import Path

def run_cmd(cmd, what):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        raise RuntimeError(f"{what} FAILED (rc={r.returncode}):\n{r.stderr}")
    return r

if MANIFEST.exists():
    print(f"Frozen manifest already present — SKIPPING refreeze: {MANIFEST}")
else:
    print(f"No frozen manifest at {MANIFEST} — running refreeze.py (stratified, dev only)...")
    run_cmd([sys.executable, str(REFREEZE), "--data", str(DATA_DIR),
             "--source-tree", str(REPO_DIR), "--out", str(MANIFEST)], "refreeze")

# Manifest gate: status + ids_sha256 recomputed + all ids dev + no held-out.
# Any failure here stops the notebook BEFORE weights are loaded.
gate = run_cmd([sys.executable, str(RUNNER), "validate-manifest",
                "--source-tree", str(REPO_DIR), "--benchmark", str(MANIFEST)], "validate-manifest")
manifest = json.loads(MANIFEST.read_text())
assert manifest["status"] == "stratified-frozen", manifest["status"]
assert len(manifest["ids"]) == manifest["n"] and manifest["n"] > 0
for cls, cnt in manifest["stratum_selected_counts"].items():
    assert cnt >= 20, f"stratum {cls} has {cnt} < 20 — refreeze rule violated."
if manifest.get("coverage_shortfall"):
    print("COVERAGE SHORTFALL recorded by refreeze (thresholds not lowered silently):",
          manifest["coverage_shortfall"])
print(f"Manifest gate PASS: n={manifest['n']} status=stratified-frozen "
      f"strata={manifest['stratum_selected_counts']} coverage={manifest['coverage']}")
print(f"Pilot 20 = first 20 ids of the manifest, in manifest order (ids not printed here).")

# Input size re-check against the manifest, still before weights.
wavs = sorted((DATA_DIR / "train").rglob("*.wav"))
assert len(wavs) >= manifest["n"], "fewer wav files than manifest samples — STOP."
gate_report = {"manifest": str(MANIFEST), "n": manifest["n"],
               "ids_sha256": manifest["ids_sha256"],
               "stratum_selected_counts": manifest["stratum_selected_counts"],
               "coverage": manifest["coverage"],
               "coverage_shortfall": manifest.get("coverage_shortfall", []),
               "wav_files": len(wavs), "weights_loaded": False}
(OUT / "manifest_gate.json").write_text(json.dumps(gate_report, indent=2), encoding="utf-8")
print("Manifest gate report ->", OUT / "manifest_gate.json")
print("Weights may now be loaded, starting with the ASR cache cell.")


In [ ]:
# Cell 3 — ASR cache: each wav transcribed exactly once per checkpoint.
# Gold transcripts never enter this cache. This is the first weight load
# (ASR only); it runs after preflight + manifest gate, never before.
import json, subprocess, sys

def run_cmd(cmd, what):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        raise RuntimeError(f"{what} FAILED (rc={r.returncode}):\n{r.stderr}")
    return r

asr_cache_cmd = [sys.executable, str(RUNNER), "asr-cache",
         "--source-tree", str(REPO_DIR), "--data", str(DATA_DIR),
         "--benchmark", str(MANIFEST), "--out", str(OUT),
         "--asr-backend", ASR_BACKEND]
if ASR_MODEL_DIR is not None:
    asr_cache_cmd += ["--asr-model-dir", str(ASR_MODEL_DIR)]
run_cmd(asr_cache_cmd, "asr-cache")
cache = json.loads((OUT / "asr_cache_baseline.json").read_text())
print(f"ASR cache: checkpoint={cache['meta']['checkpoint']} backend={cache['meta']['backend']} "
      f"entries={len(cache.get('entries', {}))}")
print("Cache key includes checkpoint + wav sha; cross-checkpoint reuse is refused by the runner.")
print("Gold transcripts were not passed to transcribe() in this cell.")


In [ ]:
# Cell 4 — Run B0 (baseline ASR, current turn only).
# Writes traces_B0.jsonl to OUT. No score is computed in this cell.
import json, subprocess, sys

def run_cmd(cmd, what):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        raise RuntimeError(f"{what} FAILED (rc={r.returncode}):\n{r.stderr}")
    return r

run_cmd_b0 = [sys.executable, str(RUNNER), "run",
         "--source-tree", str(REPO_DIR), "--data", str(DATA_DIR),
         "--benchmark", str(MANIFEST), "--mode", "B0", "--out", str(OUT),
         "--asr-backend", ASR_BACKEND, "--resume"]
if ASR_MODEL_DIR is not None:
    run_cmd_b0 += ["--asr-model-dir", str(ASR_MODEL_DIR)]
run_cmd(run_cmd_b0, "run B0")
# The runner writes run_manifest_B0.json itself (per-mode, with status +
# input hashes); --resume reuses only traces whose input hash matches.
b0_manifest = json.loads((OUT / "run_manifest_B0.json").read_text())
assert b0_manifest["status"] == "COMPLETE", (
    f"B0 run INCOMPLETE: {b0_manifest} — fix the run (resume / re-run) "
    "before scoring; score will refuse a partial run.")
print("B0 traces ->", OUT / "traces_B0.jsonl")
print("B0 run manifest (authoritative, per-mode):", OUT / "run_manifest_B0.json")


In [ ]:
# Cell 5 — Run B-gold (gold transcript as current-turn input, real AgentLoop).
# Same samples, same order, same LLM, same config as B0. Gold tool calls
# are NOT read here and are NOT input or history; only the gold transcript
# (audio_text) is the agent's current-turn input. No score is computed here.
import json, subprocess, sys

def run_cmd(cmd, what):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        raise RuntimeError(f"{what} FAILED (rc={r.returncode}):\n{r.stderr}")
    return r

run_cmd_bg = [sys.executable, str(RUNNER), "run",
         "--source-tree", str(REPO_DIR), "--data", str(DATA_DIR),
         "--benchmark", str(MANIFEST), "--mode", "B-gold", "--out", str(OUT),
         "--asr-backend", ASR_BACKEND, "--resume"]
if ASR_MODEL_DIR is not None:
    run_cmd_bg += ["--asr-model-dir", str(ASR_MODEL_DIR)]
run_cmd(run_cmd_bg, "run B-gold")
bg_manifest = json.loads((OUT / "run_manifest_B-gold.json").read_text())
assert bg_manifest["status"] == "COMPLETE", (
    f"B-gold run INCOMPLETE: {bg_manifest} — fix the run before scoring.")
print("B-gold traces ->", OUT / "traces_B-gold.jsonl")
print("Both run traces are now on disk. The score cell is the first place gold calls are read.")


In [ ]:
# Cell 6 — OFF-vs-ON check (shadow observer must not change behaviour).
# Pilot 20, condition B0, observer OFF then ON. Submission lines
# (tool_calls + state_hash) must be identical. If they differ, this cell
# fails and the main traces must be treated as INVALID — do not score them.
import json, subprocess, sys

def run_cmd(cmd, what):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        raise RuntimeError(f"{what} FAILED (rc={r.returncode}):\n{r.stderr}")
    return r

off_on_cmd = [sys.executable, str(RUNNER), "check-off-on",
         "--source-tree", str(REPO_DIR), "--data", str(DATA_DIR),
         "--benchmark", str(MANIFEST), "--out", str(OUT / "off_on"),
         "--asr-backend", ASR_BACKEND]
if ASR_MODEL_DIR is not None:
    off_on_cmd += ["--asr-model-dir", str(ASR_MODEL_DIR)]
run_cmd(off_on_cmd, "check-off-on")
report = json.loads((OUT / "off_on" / "off_on_report.json").read_text())
assert report["identical"], f"OFF-vs-ON MISMATCH: {report} — main traces are INVALID, do not score."
print(f"OFF-vs-ON PASS: {report['n_common']} pilot samples, submission lines identical.")


In [ ]:
# Cell 7 — Score + attribution + bootstrap CI (score phase ONLY).
# This cell reads run traces from disk FIRST, and only then reads gold
# calls to compute scores. It never runs the agent. It refuses to score
# unless the OFF-vs-ON report says identical.
import json, subprocess, sys

off_on = json.loads((OUT / "off_on" / "off_on_report.json").read_text())
assert off_on["identical"], "OFF-vs-ON did not pass — refusing to score."

def run_cmd(cmd, what):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-4000:])
    if r.returncode != 0:
        raise RuntimeError(f"{what} FAILED (rc={r.returncode}):\n{r.stderr}")
    return r

run_cmd([sys.executable, str(RUNNER), "score",
         "--traces", str(OUT), "--source-tree", str(REPO_DIR),
         "--data", str(DATA_DIR), "--benchmark", str(MANIFEST),
         "--out", str(OUT)], "score")
gate = json.loads((OUT / "completeness_gate.json").read_text())
assert gate["status"] == "COMPLETE", (
    f"Score gate INCOMPLETE: {gate} — the runner must not have printed a total.")
report = json.loads((OUT / "score_report.json").read_text())
print("=== Tier 1a result (official-style via lifemate/scoring.py on dev, NOT a BTC score) ===")
print(f"n={report['n']} mean B-gold={report['mean_score_B-gold']:.4f} "
      f"mean B0={report['mean_score_B0']:.4f} "
      f"(reweighted dev: B-gold={report['mean_score_B-gold_reweighted_dev']:.4f} "
      f"B0={report['mean_score_B0_reweighted_dev']:.4f})")
print(f"paired diff B-gold - B0 = {report['paired_diff_B-gold_minus_B0']}")
print(f"pure LLM error (1 - B-gold) = {report['pure_llm_error_1_minus_B-gold']:.4f}")
print(f"WER current = {report['wer_current']} slot_error_rate = {report['slot_error_rate']}")
print(f"attribution = {report['attribution_B-gold_vs_B0']}")
print(f"decision ({report['decision_rule']}): {report['decision']}")
print("Decision words are exactly the pre-registered ones: direction_concluded / "
      "not_separable / inconclusive. A diff < 0.02 or wrong-both share > 0.50 "
      "means 'not separable', never a win claim.")


In [ ]:
# Cell 8 — List output artifacts (all under /kaggle/working only).
# Nothing in this cell writes to the repo. Sensitive outputs (data,
# transcripts, traces, weights are NOT stored here — weights live in
# the Kaggle/HF cache, traces live here) stay in the working directory.
import json, subprocess
from pathlib import Path

total = 0
rows = []
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        rows.append((str(p.relative_to(WORKING)), p.stat().st_size))
        total += p.stat().st_size
for rel, size in rows:
    print(f"{size:>12}  {rel}")
print(f"{'TOTAL':>12}  {total} bytes across {len(rows)} files in {OUT}")
print("Expected artifacts: preflight_report.json, dependency_report.json, "
      "manifest_gate.json, config_snapshot.json, run_manifest_B0.json, "
      "run_manifest_B-gold.json, excluded.json, asr_cache_baseline.json, "
      "asr_cache_log.json, model_load.json, traces_B0.jsonl, traces_B-gold.jsonl, "
      "off_on/off_on_report.json, completeness_gate.json, "
      "scored_traces_B0.jsonl, scored_traces_B-gold.jsonl, score_report.json.")
print("Repo status (should be clean — notebook outputs never go to the repo):")
r = subprocess.run(["git", "-C", str(REPO_DIR), "status", "--short"],
                   capture_output=True, text=True)
print(r.stdout if r.stdout.strip() else "(clean)")
print("DONE. Download the working output from Kaggle as the private run artifact; "
      "do not commit it to GitHub.")
